In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader('../data/securely.pdf')
docs = loader.load()

In [4]:
len(docs)

59

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs =  splitter.split_documents(docs)

In [7]:
len(splitted_docs)

113

In [8]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

In [10]:
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [12]:
same_record = vector_store.similarity_search('usecase description')

In [13]:
len(same_record)

4

In [31]:
@tool
def retriever_tool(query:str):
    """
        This tool can help ypu tp retrieve the relevant data of the pdf documents.
    """
    print('Tool called for: ', query)
    docs = vector_store.similarity_search(query=query, k=4)
    context = ''

    for doc in docs:
        context += doc.page_content + '\n\n'

    return context


In [18]:
llm = ChatOpenAI(model='gpt-5-nano-2025-08-07')

In [ ]:
system_prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the 'retriever_tool' tool for for questions requiring external knowledge
"""

In [33]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [37]:
query = "does securely contain Screenshot Detection feature, and does securely contain User Manual? "

In [38]:
res = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': query
            }
        ]
    }
)

Tool called for:  Securely app screenshot detection feature user manual
Tool called for:  Securely app user manual


In [39]:
result = res['messages'][-1].content

In [40]:
print(result)

Short answer: Yes to both.

What the docs say

- Screenshot Detection feature:
  - Yes, Securely includes a Screenshot Detection capability.
  - How it works: When a user takes a screenshot, the native layer detects it (iOS: UIApplicationUserDidTakeScreenshotNotification; Android: detection of a file write to media storage). It sends an event over an EventChannel ('securely/onScreenshot') to the host app, which can respond (e.g., log out the user, flag a transaction, clear cache).
  - Notes: If Android storage permissions are restricted, it can fall back to lifecycle focus monitoring or showing a warning. The alert/event stream is designed to deliver within about 150 ms of the screenshot.

- User Manual:
  - Yes, Securely provides a User Manual.
  - It’s included as Chapter 5: USER MANUAL, with sections on Integration Setup, Code Examples & Customization, Scenario descriptions, Testing, Deployment, etc.
  - The Integration Setup section covers adding the package, platform configuration